## Uni-Mol tools

- Source: https://github.com/deepmodeling/Uni-Mol (folder `unimol_tools`)
- Citation: Zhou, G. *et al.* Uni-Mol: A Universal 3D Molecular Representation Learning Framework. in *The Eleventh International Conference on Learning Representations* (2023).

Setup: download the Uni-Mol repository, rename the repository folder to `unimol`, and put it in the root of this repository. The notebook imports the package in `unimol/unimol_tools/unimol_tools`, and the Uni-Mol dictionary is downloaded on the first run. Install the dependencies of Uni-Mol tools with pip:

```
pip install addict huggingface_hub joblib pyyaml numba lightning torchmetrics rdkit
```

In [ ]:
import gc
import os

import numpy as np
import pandas as pd
import torch
from torch import nn
from unimol.unimol_tools.unimol_tools.data.conformer import coords2unimol
from unimol.unimol_tools.unimol_tools.data.dictionary import Dictionary
from unimol.unimol_tools.unimol_tools.models.transformers import (
    TransformerEncoderWithPair,
)
from unimol.unimol_tools.unimol_tools.models.unimol import (
    GaussianLayer,
    LinearHead,
    NonLinearHead,
    molecule_architecture,
)
from unimol.unimol_tools.unimol_tools.utils import pad_1d_tokens, pad_2d, pad_coords
from unimol.unimol_tools.unimol_tools.weights import WEIGHT_DIR

# Restore HF_ENDPOINT, which Uni-Mol weighthub sets to mirror at import
os.environ["HF_ENDPOINT"] = "https://huggingface.co"

import warnings

from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger
from torch.utils.data import DataLoader, Dataset

from benchmarks import (
    BestEpochTracker,
    LightningModule,
    MultiTargetScaler,
    PrintEpochMetricsCallback,
)

warnings.filterwarnings("ignore", message=".*does not have many workers.*")

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
FOLDS = range(1, 6)

target_properties = [
    "xtb_homo",
    "xtb_lumo",
    "xtb_hl_gap",
    "pm7_homo",
    "pm7_lumo",
    "pm7_hl_gap",
    "pm7_hof",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "extensive",
]


PRETRAINED = False
USE_DEFAULT_ARCH = False

# REMOVE_HS = True selects no-H checkpoint mol_pre_no_h_220816.pt
REMOVE_HS = True

# Intensive targets from [CLS] token, extensive pm7_hof summed over atom tokens
DUAL_READOUT = True

TAG = "unimol_1.1M-conformer"
model_path_template = "../models/" + TAG + "-[PAC5-4]-fold_{fold}"
model_logs_dir = "../logs"
model_dataset_dir = "../dataset"
model_logs_template = TAG + "-[PAC5-4]-fold_{fold}"

SPLIT_DIR = model_dataset_dir + "/PAC5-4"

MAX_ATOMS = 256  # Above PAC5-4 maximum
BATCH_SIZE = 256
EMBED_DIM = 176

MAX_EPOCHS = 500
ES_PATIENCE = 50
LR_PATIENCE = 25
MIN_LR = 1e-6
LR = 1e-3  # Parity with other models
NUM_WORKERS = 0

# Built by prep_3d_geometry_cache.py
_geom = np.load(SPLIT_DIR + "/geom_openff.npz")
GEOM_Z, GEOM_R, GEOM_PTR = _geom["Z"], _geom["R"], _geom["ptr"]
_gidx = pd.read_csv(SPLIT_DIR + "/geom_openff_index.csv")
SMILES_TO_ROW = dict(zip(_gidx["smiles"].values, _gidx["row"].values))
print(f"Geometry cache: {len(GEOM_PTR) - 1} molecules, {len(GEOM_Z)} atoms")
del _geom, _gidx

Z_TO_SYMBOL = {
    1: "H",
    5: "B",
    6: "C",
    7: "N",
    8: "O",
    9: "F",
    14: "Si",
    15: "P",
    16: "S",
    17: "Cl",
    35: "Br",
    53: "I",
}

# Uni-Mol molecular dictionary: 30 symbols plus [MASK]
from unimol.unimol_tools.unimol_tools.config import MODEL_CONFIG
from unimol.unimol_tools.unimol_tools.weights import weight_download

_dict_name = MODEL_CONFIG["dict"]["molecule_no_h" if REMOVE_HS else "molecule_all_h"]
if not os.path.exists(os.path.join(WEIGHT_DIR, _dict_name)):
    weight_download(_dict_name, WEIGHT_DIR)
DICTIONARY = Dictionary.load(os.path.join(WEIGHT_DIR, _dict_name))
DICTIONARY.add_symbol("[MASK]", is_special=True)
print(f"Dictionary: {len(DICTIONARY)} symbols")


class PlateauLightningModule(LightningModule):
    """LightningModule from benchmarks with ReduceLROnPlateau instead of OneCycleLR."""

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.lr, weight_decay=self.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=LR_PATIENCE, min_lr=MIN_LR
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1,
            },
        }

In [ ]:
class PAC5UniMolDataset(Dataset):
    """PAC5 molecules featurised with coords2unimol"""

    def __init__(
        self, smiles, scaled_targets, remove_hs=REMOVE_HS, max_atoms=MAX_ATOMS
    ):
        rows, keep = [], []
        for i, s in enumerate(smiles):
            r = SMILES_TO_ROW.get(s)
            if r is not None:
                rows.append(r)
                keep.append(i)
        self.rows = np.asarray(rows, dtype=np.int64)
        self.y = np.asarray(scaled_targets, dtype=np.float32)[keep]
        self.n_dropped = len(smiles) - len(self.rows)
        self.remove_hs = remove_hs
        self.max_atoms = max_atoms

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        s, e = GEOM_PTR[row], GEOM_PTR[row + 1]
        z = GEOM_Z[s:e]
        coords = np.ascontiguousarray(GEOM_R[s:e]).astype(np.float32)
        atoms = [Z_TO_SYMBOL[int(a)] for a in z]
        feats = coords2unimol(
            atoms,
            coords,
            DICTIONARY,
            max_atoms=self.max_atoms,
            remove_hs=self.remove_hs,
        )
        return feats, self.y[index]


def make_collate(padding_idx):
    """Uni-Mol's batch_collate_fn, lifted off the model, returning (inputs, y)"""

    def _collate(samples):
        batch = {}
        for k in samples[0][0].keys():
            if k == "src_coord":
                v = pad_coords(
                    [torch.tensor(s[0][k]).float() for s in samples], pad_idx=0.0
                )
            elif k == "src_edge_type":
                v = pad_2d(
                    [torch.tensor(s[0][k]).long() for s in samples], pad_idx=padding_idx
                )
            elif k == "src_distance":
                v = pad_2d(
                    [torch.tensor(s[0][k]).float() for s in samples], pad_idx=0.0
                )
            elif k == "src_tokens":
                v = pad_1d_tokens(
                    [torch.tensor(s[0][k]).long() for s in samples], pad_idx=padding_idx
                )
            else:
                raise KeyError(f"Unexpected Uni-Mol input feature: {k}")
            batch[k] = v
        y = torch.tensor(np.stack([s[1] for s in samples]), dtype=torch.float32)
        return batch, y

    return _collate


def scaled_architecture(encoder_layers=3, embed_dim=176, ffn_ratio=4, heads=None):
    """Uni-Mol's molecule architecture at reduced depth and width.

    ffn_embed_dim / embed_dim = 4 and head_dim = 8 are kept from the stock 512/2048/64.
    """
    args = molecule_architecture()
    args.encoder_layers = encoder_layers
    args.encoder_embed_dim = embed_dim
    args.encoder_ffn_embed_dim = embed_dim * ffn_ratio
    args.encoder_attention_heads = (
        heads if heads is not None else max(1, embed_dim // 8)
    )
    return args


class UniMolMultiTask(nn.Module):
    """Uni-Mol backbone with overridable architecture args and optional pretrained weights."""

    def __init__(
        self,
        property_types,
        dictionary,
        args,
        dual_readout=DUAL_READOUT,
        pretrain_path=None,
    ):
        super().__init__()
        self.args = args
        self.dictionary = dictionary
        self.property_types = list(property_types)
        self.n_int = self.property_types.count("intensive")
        self.n_ext = self.property_types.count("extensive")
        self.dual_readout = dual_readout and self.n_ext > 0
        self.padding_idx = dictionary.pad()

        self.embed_tokens = nn.Embedding(
            len(dictionary), args.encoder_embed_dim, self.padding_idx
        )
        self.encoder = TransformerEncoderWithPair(
            encoder_layers=args.encoder_layers,
            embed_dim=args.encoder_embed_dim,
            ffn_embed_dim=args.encoder_ffn_embed_dim,
            attention_heads=args.encoder_attention_heads,
            emb_dropout=args.emb_dropout,
            dropout=args.dropout,
            attention_dropout=args.attention_dropout,
            activation_dropout=args.activation_dropout,
            max_seq_len=args.max_seq_len,
            activation_fn=args.activation_fn,
            no_final_head_layer_norm=args.delta_pair_repr_norm_loss < 0,
        )
        K = 128
        n_edge_type = len(dictionary) * len(dictionary)
        self.gbf_proj = NonLinearHead(
            K, args.encoder_attention_heads, args.activation_fn
        )
        self.gbf = GaussianLayer(K, n_edge_type)

        n_cls_out = self.n_int if self.dual_readout else len(self.property_types)
        self.classification_head = LinearHead(
            input_dim=args.encoder_embed_dim,
            num_classes=n_cls_out,
            pooler_dropout=args.pooler_dropout,
        )
        if self.dual_readout:
            self.extensive_norm = nn.BatchNorm1d(args.encoder_embed_dim)
            self.extensive_head = LinearHead(
                input_dim=args.encoder_embed_dim,
                num_classes=self.n_ext,
                pooler_dropout=args.pooler_dropout,
            )

        if pretrain_path is not None:
            state = torch.load(pretrain_path, map_location="cpu")
            state = state.get("model", state)
            missing, unexpected = self.load_state_dict(state, strict=False)
            print(
                f"Loaded pretrained weights: {len(missing)} missing, "
                f"{len(unexpected)} unexpected keys"
            )

    def forward(self, batch):
        src_tokens = batch["src_tokens"]
        src_distance = batch["src_distance"]
        src_edge_type = batch["src_edge_type"]

        padding_mask = src_tokens.eq(self.padding_idx)
        if not padding_mask.any():
            padding_mask = None
        x = self.embed_tokens(src_tokens)

        n_node = src_distance.size(-1)
        gbf_feature = self.gbf(src_distance, src_edge_type)
        graph_attn_bias = self.gbf_proj(gbf_feature)
        graph_attn_bias = graph_attn_bias.permute(0, 3, 1, 2).contiguous()
        graph_attn_bias = graph_attn_bias.view(-1, n_node, n_node)

        encoder_rep, _, _, _, _ = self.encoder(
            x, padding_mask=padding_mask, attn_mask=graph_attn_bias
        )

        cls_repr = encoder_rep[:, 0, :]
        cls_out = self.classification_head(cls_repr)

        if not self.dual_readout:
            return cls_out

        atom_mask = (
            (
                src_tokens.ne(self.padding_idx)
                & src_tokens.ne(self.dictionary.bos())
                & src_tokens.ne(self.dictionary.eos())
            )
            .unsqueeze(-1)
            .to(encoder_rep.dtype)
        )
        ext_repr = (encoder_rep * atom_mask).sum(dim=1)
        ext_out = self.extensive_head(self.extensive_norm(ext_repr))

        out, i, e = [], 0, 0
        for p in self.property_types:
            if p == "intensive":
                out.append(cls_out[:, i : i + 1])
                i += 1
            else:
                out.append(ext_out[:, e : e + 1])
                e += 1
        return torch.cat(out, dim=1)


def build_model():
    """Returns (model, description)"""
    if PRETRAINED:
        name = "molecule_" + ("no_h" if REMOVE_HS else "all_h")
        wpath = os.path.join(WEIGHT_DIR, MODEL_CONFIG["weight"][name])
        if not os.path.exists(wpath):
            weight_download(MODEL_CONFIG["weight"][name], WEIGHT_DIR)
        args = molecule_architecture()  # 15 layers, 512 dim, as published
        m = UniMolMultiTask(property_types, DICTIONARY, args, pretrain_path=wpath)
        return (
            m,
            f"Pretrained, {args.encoder_layers} layers, dim {args.encoder_embed_dim}",
        )
    args = (
        molecule_architecture()
        if USE_DEFAULT_ARCH
        else scaled_architecture(encoder_layers=3, embed_dim=EMBED_DIM)
    )
    m = UniMolMultiTask(property_types, DICTIONARY, args, pretrain_path=None)
    return (
        m,
        f"From scratch, {args.encoder_layers} layers, dim {args.encoder_embed_dim}",
    )

In [ ]:
fold_results = []
best_epochs = []
stopped_epochs = []

for fold in FOLDS:
    if fold in [4, 5]:
        pass
    else:
        continue
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    train_df = pd.read_csv(SPLIT_DIR + f"/fold_{fold}_train.csv").dropna(
        subset=target_properties
    )
    val_df = pd.read_csv(SPLIT_DIR + f"/fold_{fold}_val.csv").dropna(
        subset=target_properties
    )

    train_targets = train_df[target_properties].values.astype(np.float32)
    val_targets = val_df[target_properties].values.astype(np.float32)

    scaler = MultiTargetScaler()
    scaler.fit(train_targets)

    train_dataset = PAC5UniMolDataset(
        train_df["smiles"].values, scaler.transform(train_targets)
    )
    val_dataset = PAC5UniMolDataset(
        val_df["smiles"].values, scaler.transform(val_targets)
    )
    print(
        f"Train {len(train_dataset)} (dropped {train_dataset.n_dropped}) | "
        f"Val {len(val_dataset)} (dropped {val_dataset.n_dropped})"
    )

    model, desc = build_model()
    print(
        f"Model: {desc} | {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parameters"
    )

    coll = make_collate(model.padding_idx)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=coll,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        persistent_workers=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=coll,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        persistent_workers=False,
    )

    lightning_model = PlateauLightningModule(model, target_properties, scaler, lr=LR)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=ES_PATIENCE,
        min_delta=0.0,
        check_on_train_epoch_end=False,
        verbose=False,
    )
    best_epoch_tracker = BestEpochTracker()

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[
            checkpoint_callback,
            early_stop,
            best_epoch_tracker,
            PrintEpochMetricsCallback(target_properties),
        ],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    best_epoch, last_epoch = best_epoch_tracker.best_epoch, best_epoch_tracker.last_epoch
    if best_epoch is None:
        raise RuntimeError("The best-epoch tracker did not record a validation epoch.")
    best_epochs.append(best_epoch)
    stopped_epochs.append(last_epoch)
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    print(f" + Best epoch {best_epoch} of {last_epoch} run (ceiling {MAX_EPOCHS})")
    fold_results.append(best_val_loss)

    best_lightning_model = PlateauLightningModule.load_from_checkpoint(
        checkpoint_callback.best_model_path,
        model=model,
        property_names=target_properties,
        scaler=scaler,
    )
    torch.save(best_lightning_model.model.state_dict(), save_path_dir + "/best.ckpt")

    lightning_model.cpu()
    best_lightning_model.cpu()

    del train_df, val_df, train_dataset, val_dataset, train_loader, val_loader
    del (
        model,
        lightning_model,
        best_lightning_model,
        trainer,
        checkpoint_callback,
        logger,
        scaler,
    )
    del early_stop, best_epoch_tracker

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
print(f"\nBest-validation epoch per fold: {best_epochs} (ran {stopped_epochs})")
print(
    f"Latest best-validation epoch: {max(best_epochs)} of {MAX_EPOCHS}"
)

In [ ]:
metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {prop: {m: [] for m in metric_keys} for prop in target_properties},
    "test": {prop: {m: [] for m in metric_keys} for prop in target_properties},
}

for fold in FOLDS:
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    train_df = pd.read_csv(SPLIT_DIR + f"/fold_{fold}_train.csv").dropna(
        subset=target_properties
    )
    test_df = pd.read_csv(SPLIT_DIR + f"/fold_{fold}_test.csv").dropna(
        subset=target_properties
    )

    train_targets = train_df[target_properties].values.astype(np.float32)
    test_targets = test_df[target_properties].values.astype(np.float32)

    scaler = MultiTargetScaler()
    scaler.fit(train_targets)

    train_dset = PAC5UniMolDataset(
        train_df["smiles"].values, scaler.transform(train_targets)
    )
    test_dset = PAC5UniMolDataset(
        test_df["smiles"].values, scaler.transform(test_targets)
    )

    model, desc = build_model()
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    model.load_state_dict(torch.load(load_path, map_location=device))

    coll = make_collate(model.padding_idx)
    train_loader = DataLoader(
        train_dset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=coll,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        persistent_workers=False,
    )
    test_loader = DataLoader(
        test_dset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=coll,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        persistent_workers=False,
    )

    lightning_model = PlateauLightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    print("\n======[ Train set ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("\n======[ Test set ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for i, prop in enumerate(target_properties):
        print(f"======[ {prop} ]======")
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        all_results["train"][prop]["mae"].append(t_mae)
        all_results["train"][prop]["rmsd"].append(t_rmsd)
        all_results["train"][prop]["max"].append(t_max)
        all_results["train"][prop]["min"].append(t_min)
        all_results["train"][prop]["r2"].append(t_r2)

        all_results["test"][prop]["mae"].append(te_mae)
        all_results["test"][prop]["rmsd"].append(te_rmsd)
        all_results["test"][prop]["max"].append(te_max)
        all_results["test"][prop]["min"].append(te_min)
        all_results["test"][prop]["r2"].append(te_r2)

    del model, lightning_model, trainer, scaler, train_results, test_results
    del train_loader, test_loader, train_dset, test_dset, train_df, test_df
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"======[ Cross-validation results: Uni-mol {TAG} ]======")

ddof = 1 if len(FOLDS) > 1 else 0
for prop in target_properties:
    print(f"\n======[ {prop} ]======")
    t_stats = {
        m: (
            np.mean(all_results["train"][prop][m]),
            np.std(all_results["train"][prop][m], ddof=ddof),
        )
        for m in metric_keys
    }
    print(
        f" + Train -> "
        f"MAE: {t_stats['mae'][0]:.4f} ± {t_stats['mae'][1]:.4f}\t"
        f"RMSD: {t_stats['rmsd'][0]:.4f} ± {t_stats['rmsd'][1]:.4f}\t"
        f"Max: {t_stats['max'][0]:.4f} ± {t_stats['max'][1]:.4f}\t"
        f"Min: {t_stats['min'][0]:.4f} ± {t_stats['min'][1]:.4f}\t"
        f"R^2: {t_stats['r2'][0]:.4f} ± {t_stats['r2'][1]:.4f}"
    )

    te_stats = {
        m: (
            np.mean(all_results["test"][prop][m]),
            np.std(all_results["test"][prop][m], ddof=ddof),
        )
        for m in metric_keys
    }
    print(
        f" + Test -> "
        f"MAE: {te_stats['mae'][0]:.4f} ± {te_stats['mae'][1]:.4f}\t"
        f"RMSD: {te_stats['rmsd'][0]:.4f} ± {te_stats['rmsd'][1]:.4f}\t"
        f"Max: {te_stats['max'][0]:.4f} ± {te_stats['max'][1]:.4f}\t"
        f"Min: {te_stats['min'][0]:.4f} ± {te_stats['min'][1]:.4f}\t"
        f"R^2: {te_stats['r2'][0]:.4f} ± {te_stats['r2'][1]:.4f}"
    )